## Reference vs Whisper Generated Transcripts Audit

In [1]:
import pandas as pd
import torch

In [10]:
reference_inputs = torch.load('embeddings/linguistic/distilbert_reference/distilbert_reference_train.pt')
reference_uids = pd.Series([i[0] for i in reference_inputs], name = 'SNo')
train_manifest = pd.read_csv('results/artifacts/train_manifest.csv')
train_manifest = train_manifest[['SNo', 'Reference_Transcript']]
merged = pd.merge(reference_uids, train_manifest, on='SNo', how='inner')
merged = merged.drop_duplicates()
reference = merged['Reference_Transcript'].reset_index(drop=True)

whisper = pd.read_csv('results/artifacts/whisper_transcripts_train.csv')['Whisper_Transcript'].reset_index(drop = True)
whisper = whisper.astype(str).str.strip().replace(['', 'nan', 'NaN', 'None'], None)
whisper = whisper.dropna()

In [13]:
total_len = len(reference)
print("total samples: ", total_len)

total samples:  2778


### Number of Usable Transcripts

In [11]:
reference_usable = reference.str.strip().ne("").sum()
whisper_usable = whisper.str.strip().ne("").sum()

print(f"number of usable samples in reference: {reference_usable}")
print(f"number of usable samples in whisper: {whisper_usable}")

if whisper_usable < reference_usable:
    print(f"\nnumber of missing transcripts in whisper: {reference_usable - whisper_usable}")

number of usable samples in reference: 2778
number of usable samples in whisper: 2677

number of missing transcripts in whisper: 101


### Coverage & Missing Rates

In [17]:
reference_cr = (reference_usable / total_len) * 100
whisper_cr = (whisper_usable / total_len) * 100

print(f"coverage rate of reference: {reference_cr:.2f}")
print(f"coverage rate of whisper: {whisper_cr:.2f}")

coverage rate of reference: 100.00
coverage rate of whisper: 96.36


In [18]:
reference_mr = 100 - reference_cr
whisper_mr = 100 - whisper_cr

print(f"missing rate of reference: {reference_mr:.2f}")
print(f"missing rate of whisper: {whisper_mr:.2f}")

missing rate of reference: 0.00
missing rate of whisper: 3.64


### Emotion Coverage by Whisper